In [18]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim

from sklearn.datasets import make_gaussian_quantiles
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import pandas as pd

import copy
import time

import pennylane as qml
from pennylane.optimize import NesterovMomentumOptimizer
from pennylane import numpy as np

In [ ]:

dev = qml.device("default.qubit")

# Parameter
# --- CONFIGURATION ---
n_samples = 100
n_features_original = 28
n_components = 5
n_qubits = 20
n_clusters_per_class = 1
evolution_time = 1.0
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)# We create our device (ideal simulator -> zero noise, infinite shots)

In [20]:
def angle_encoder(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

def hebbian_weights(patterns):
    return np.sum([np.outer(p, p) for p in patterns], axis=0)

def build_hamiltonian(W):
    H = 0
    for i in range(n_qubits):
        for j in range(n_qubits):
            if i != j:
                H += W[i, j] * qml.PauliZ(i) @ qml.PauliZ(j)
    return -H  # negative for energy minima

In [21]:
@qml.qnode(dev)
def evolve_and_measure(x, time=evolution_time):
    angle_encoder(x)
    qml.ApproxTimeEvolution(H, time, 1)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [22]:

def square_loss(labels, predictions):
    # We use a call to qml.math.stack to allow subtracting the arrays directly
    return np.mean((labels - qml.math.stack(predictions)) ** 2)


def accuracy(labels, predictions):
    acc = sum(abs(l - p) < 1e-5 for l, p in zip(labels, predictions))
    acc = acc / len(labels)
    return acc

In [166]:
data = np.loadtxt("data/iris_classes1and2_scaled.txt")
X = data[:,:4]
Y = data[:,4]
scaler = MinMaxScaler(feature_range=(-1, 1))
print(f"First training sample: {X[0]}")
X = scaler.fit_transform(X)
print(f"First training sample - normalized: {features[0]}")
hidden_data = np.random.choice([-1,1], size = X.shape[0] * 15)
hidden_data = hidden_data.reshape( ( X.shape[0], 15 ) )
Y = Y.reshape((X.shape[0], 1))
print(hidden_data.shape)
X = np.append(X,Y, axis = 1)
X = np.append(X,hidden_data, axis = 1)
print(X.shape)
print(X[0])

First training sample: [0.4  0.75 0.2  0.05]
First training sample - normalized: [-0.40740741  0.25       -0.80487805 -0.88235294]
(100, 15)
(100, 20)
[-0.40740741  0.25       -0.80487805 -0.88235294 -1.         -1.
  1.          1.          1.         -1.          1.          1.
  1.         -1.          1.          1.         -1.         -1.
  1.         -1.        ]


In [167]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(X[:, 0][Y == 1], X[:, 1][Y == 1], c="b", marker="o", ec="k")
plt.scatter(X[:, 0][Y == -1], X[:, 1][Y == -1], c="r", marker="o", ec="k")
plt.title("Original data")
plt.show()

plt.figure()
dim1 = 0
dim2 = 1
plt.scatter(features[:, dim1][Y == 1], features[:, dim2][Y == 1], c="b", marker="o", ec="k")
plt.scatter(features[:, dim1][Y == -1], features[:, dim2][Y == -1], c="r", marker="o", ec="k")
plt.title(f"Normalized data")
plt.show()

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

<Figure size 640x480 with 0 Axes>